# Eksperyment EEG XAI
Pipeline: konfiguracja → preprocessing → podział → trening → Captum → wykresy.
Logika siedzi w `src/eeg_xai/`, a tutaj tylko sterowanie. Rodzaj uczenia wybiera komórka 4.

In [ ]:
# Colab: środowisko (uruchom tylko na Colabie, lokalnie pomiń)
from google.colab import drive

drive.mount("/content/drive")
!git clone https://github.com/TWOJ_LOGIN/eeg-xai.git /content/eeg-xai
%cd /content/eeg-xai
%pip install -q -e .

In [ ]:
# Autoreload (zmiany w src/ bez restartu kernela)
%load_ext autoreload
%autoreload 2

In [ ]:
# Importy
import os
from dataclasses import replace

import joblib

from eeg_xai.data import (
    create_train_test_split,
    get_channel_names_from_header,
    load_preprocessed_patient_data,
)
from eeg_xai.explain import compute_captum_analysis, extract_global_heatmap_data
from eeg_xai.models import TemporalTransformer
from eeg_xai.plotting.animations import generate_type_3_gif, generate_type_3_topoplot_gif
from eeg_xai.plotting.heatmaps import (
    plot_difference_heatmap_and_bars,
    plot_global_heatmap_and_bars,
    plot_top_n_heatmap_and_bars,
    plot_type_2_electrode_profiles,
)
from eeg_xai.preprocessing import run_preprocessing
from eeg_xai.presets import PRESETS
from eeg_xai.training import (
    EEGLightningWrapper,
    get_best_model_checkpoint,
    train_cross_individual_torch,
)
from eeg_xai.tsp import RiemannianTokenizer
from eeg_xai.utils import get_device

In [ ]:
# 1. KONFIGURACJA: tu wybierasz rodzaj uczenia
# Gotowe warianty: "raw" | "filtered" | "filtered_motor" | "tsp"
cfg = PRESETS["filtered"]()

# Modyfikacje pojedynczych ustawień (konfiguracja jest niezmienna, więc tworzymy kopię):
# cfg = replace(cfg, train=replace(cfg.train, lr=1e-4, batch_size=32))
# cfg = replace(cfg, model=replace(cfg.model, d_model=128, n_head=8))
# cfg = replace(cfg, preprocess=replace(cfg.preprocess, l_freq=8, h_freq=30))
# Oba zbiory naraz:
# from eeg_xai.presets import both_datasets
# cfg = both_datasets(cfg)

DEVICE = get_device()
print(cfg.describe())
print(f"Device: {DEVICE}")
print("Config saved to:", cfg.save())

In [ ]:
# 2. Preprocessing i wczytanie danych (dla każdego zbioru z cfg.datasets)
MODEL_CLASS = TemporalTransformer

loaded_data = {}
target_time_pts = None  # pilnuje, żeby długość segmentu zgadzała się między zbiorami
for ds in cfg.datasets:
    run_preprocessing(cfg, ds)
    data, time_array, t0_idx, target_time_pts = load_preprocessed_patient_data(
        data_dir=str(cfg.preprocessed_dir(ds)),
        segment_type=cfg.preprocess.segment_type,
        prefix=ds.prefix,
        global_expected_time_points=target_time_pts,
    )
    loaded_data.update(data)

In [ ]:
# 3. Podział: trening / globalny zbiór testowy
training_subject_data, test_loader = create_train_test_split(
    subject_data=loaded_data,
    save_dir=str(cfg.global_test_dir),
    test_patient_count=cfg.train.test_patient_count,
    batch_size=cfg.train.batch_size,
    test_subjects_list=list(cfg.train.test_subjects) if cfg.train.test_subjects else None,
    sequence_first=cfg.sequence_first,
    feature_method=cfg.representation.feature_method,
)

In [ ]:
# 4. Reprezentacja wejścia (raw / TSP) i rozmiar wejścia modelu
rep = cfg.representation

if rep.feature_method == "raw":
    sample_subj = list(training_subject_data.keys())[0]
    sample_x = training_subject_data[sample_subj][0]

    num_channels = sample_x.shape[1]
    num_time_points = sample_x.shape[2]

    model_input_size = num_channels if rep.temporal_domain else num_time_points

elif rep.feature_method == "tsp":
    # Window size and sliding step
    tokenizer = RiemannianTokenizer(
        window_size_sec=rep.tsp_window_sec,
        step_size_sec=rep.tsp_step_sec,
        sfreq=float(cfg.sfreq),
    )

    # Overwrites the dictionary and the test dataset inside test_loader
    training_subject_data, _ = tokenizer.fit_transform_all(
        training_subject_data,
        test_loader.dataset,
    )

    # Model now expects 1D TSP features per sequence step
    model_input_size = tokenizer.get_feature_dim()

else:
    raise ValueError(f"Unsupported feature_method: {rep.feature_method}")

print(f"Model input size based on data shapes: {model_input_size}")

In [ ]:
# 5. Trening (lub wczytanie najlepszego checkpointu)
tr = cfg.train
explicit_path = tr.explicit_model_path

if explicit_path and os.path.exists(explicit_path):
    print("\n" + "=" * 60)
    print("BYPASSING TRAINING: Explicit model path provided!")
    print(f"Path: {explicit_path}")
    print("=" * 60 + "\n")
    best_ckpt_path = explicit_path
else:
    if tr.train_model:
        train_cross_individual_torch(
            subject_data=training_subject_data,
            model_class=MODEL_CLASS,
            epochs=tr.epochs,
            batch_size=tr.batch_size,
            lr=tr.lr,
            d_model=cfg.model.d_model,
            nhead=cfg.model.n_head,
            dataset_name=cfg.experiment_name,
            training_patience=tr.patience,
            sequence_first=cfg.sequence_first,
            noise_std=cfg.model.noise_std,
            feature_method=rep.feature_method,
            model_input_size=model_input_size,
            checkpoints_dir=str(cfg.checkpoints_dir),
        )
    else:
        print("Skipping training loop. Extracting model from saved weights.")

    best_ckpt_path = get_best_model_checkpoint(str(cfg.checkpoints_dir))

print("Loading weights into wrapper.")
best_model_wrapper = EEGLightningWrapper.load_from_checkpoint(
    checkpoint_path=best_ckpt_path,
    model_class=MODEL_CLASS,
    input_size=model_input_size,
)

best_model = best_model_wrapper.model.to(DEVICE)
best_model.eval()

In [ ]:
# 6. Explainability: Captum (Integrated Gradients)
results_high = compute_captum_analysis(
    best_model,
    test_loader,
    DEVICE,
    cfg.analysis.max_samples,
    cfg.sfreq,
    cfg.sequence_first,
    rep.feature_method,
    rep.tsp_step_sec,
)

if results_high is not None:
    # Saving captum analysis
    print(f"Saving results to {cfg.captum_path}...")
    joblib.dump(results_high, cfg.captum_path, compress=3)
    print("Save complete!")
else:
    print("Nothing to save.")

In [ ]:
# 7. Wykresy: przygotowanie (pacjenci, katalogi, nazwy kanałów)
an = cfg.analysis
tmin = cfg.preprocess.tmin

all_test_subjects = test_loader.dataset.subject_ids
patient_ids = sorted(set(all_test_subjects))
# patient_ids = ["S014", "S055", "S059", "S063", "S085"]
base_output_dir = str(cfg.plots_dir)
os.makedirs(base_output_dir, exist_ok=True)

ch_names = get_channel_names_from_header(
    str(cfg.preprocessed_dir(cfg.datasets[0])), cfg.preprocess.segment_type
)

selected = cfg.preprocess.selected_channels
if selected is not None and len(ch_names) > len(selected):
    ch_names = [ch for ch in ch_names if ch in selected]

In [ ]:
# 8. Wykres TYPU 1: globalny top-N (+ osobno lewa/prawa ręka)
global_heatmap = extract_global_heatmap_data(results_high, mode="all")
plot_top_n_heatmap_and_bars(
    global_heatmap, results_high, original_channel_names=ch_names,
    top_n=an.top_n_features, title_suffix="Global (All Subjects, All Classes)",
    save_path=os.path.join(base_output_dir, "xai_global_topN.png"),
    show_plot=False,
)

if an.plot_rl_separately:
    # Klasa 0 (Lewa ręka)
    hm_left = extract_global_heatmap_data(results_high, mode="all", target_class=0)
    plot_top_n_heatmap_and_bars(
        hm_left, results_high, original_channel_names=ch_names,
        top_n=an.top_n_features, title_suffix="Global (LEFT Hand Only)",
        save_path=os.path.join(base_output_dir, "xai_left_hand_topN.png"),
        show_plot=False,
    )

    # Klasa 1 (Prawa ręka)
    hm_right = extract_global_heatmap_data(results_high, mode="all", target_class=1)
    plot_top_n_heatmap_and_bars(
        hm_right, results_high, original_channel_names=ch_names,
        top_n=an.top_n_features, title_suffix="Global (RIGHT Hand Only)",
        save_path=os.path.join(base_output_dir, "xai_right_hand_topN.png"),
        show_plot=False,
    )

In [ ]:
# 9. Wykres TYPU 2: profile pojedynczych elektrod
if an.generate_type_2_plots:
    type2_save_dir = os.path.join(base_output_dir, "Separate_Electrode_Plots")

    # 1. Globalne dla Typu 2
    if global_heatmap is not None:
        plot_type_2_electrode_profiles(
            global_heatmap, results_high, original_channel_names=ch_names,
            top_n_electrodes=an.top_n_electrodes,
            max_top_pairs_per_plot=an.top_n_max_pairs_per_electrode,
            title_suffix="Global All", save_dir=type2_save_dir, show_plot=False,
        )

    for subj in an.target_subjects:
        hm_subj = extract_global_heatmap_data(results_high, mode="all", subject_id=subj)
        if hm_subj is not None:
            plot_type_2_electrode_profiles(
                hm_subj, results_high, original_channel_names=ch_names,
                top_n_electrodes=an.top_n_electrodes,
                max_top_pairs_per_plot=an.top_n_max_pairs_per_electrode,
                title_suffix=f"Subject {subj}", save_dir=type2_save_dir, show_plot=False,
            )

In [ ]:
# 10. Wykres TYPU 1 dla wybranych pacjentów
for subj in an.target_subjects:
    hm_subj = extract_global_heatmap_data(results_high, mode="all", subject_id=subj)

    # Jeśli ekstraktor nic nie znajdzie (np. złe ID), zwraca None
    if hm_subj is not None:
        plot_top_n_heatmap_and_bars(
            hm_subj, results_high, original_channel_names=ch_names,
            top_n=an.top_n_features, title_suffix=f"Subject: {subj}",
            save_path=os.path.join(base_output_dir, f"xai_subj_{subj}_topN.png"),
            show_plot=False,
        )

In [ ]:
# 11. Wykres TYPU 3: animacje GIF
if an.generate_type_3_gifs:
    type3_save_dir = os.path.join(base_output_dir, "Type3_Animated_GIFs")
    type3_1_save_dir = os.path.join(base_output_dir, "Type3_Animated_Topoplots")
    gif_args = dict(tmin=tmin, step_size_sec=rep.tsp_step_sec, window_size_sec=rep.tsp_window_sec)

    # 1. Global GIF
    if global_heatmap is not None:
        generate_type_3_gif(
            global_heatmap, results_high, original_channel_names=ch_names,
            title_suffix="Global All",
            save_path=os.path.join(type3_save_dir, "Global_All.gif"), **gif_args,
        )
        generate_type_3_topoplot_gif(
            global_heatmap, results_high, original_channel_names=ch_names,
            title_suffix="Global All",
            save_path=os.path.join(type3_1_save_dir, "Global_All.gif"), **gif_args,
        )

    for subj in an.target_subjects:
        hm_subj = extract_global_heatmap_data(results_high, mode="all", subject_id=subj)
        if hm_subj is not None:
            generate_type_3_gif(
                hm_subj, results_high, original_channel_names=ch_names,
                title_suffix=f"Subject {subj}",
                save_path=os.path.join(type3_save_dir, f"Subject_{subj}.gif"), **gif_args,
            )
            generate_type_3_topoplot_gif(
                hm_subj, results_high, original_channel_names=ch_names,
                title_suffix=f"Subject {subj}",
                save_path=os.path.join(type3_1_save_dir, f"Subject_{subj}.gif"), **gif_args,
            )

In [ ]:
# 12. Wykresy per pacjent: lewa / prawa / różnica
for patient in patient_ids:
    print(f"\nDrawing heatmaps for patient: {patient}")

    patient_dir = os.path.join(base_output_dir, patient)
    os.makedirs(patient_dir, exist_ok=True)

    # File names:
    left_hand_path = os.path.join(patient_dir, f"{patient}_Global_Left.png")
    right_hand_path = os.path.join(patient_dir, f"{patient}_Global_Right.png")
    diff_path = os.path.join(patient_dir, f"{patient}_Global_Difference.png")

    # Left Hand
    heatmap_left = extract_global_heatmap_data(
        results_high, mode="all", subject_id=patient, target_class=0)

    plot_global_heatmap_and_bars(
        heatmap_left, results_high,
        title_suffix=f"Patient {patient} - Left Hand Only",
        channel_names=ch_names,
        save_path=left_hand_path,
        show_plot=False,
    )

    # Right Hand
    heatmap_right = extract_global_heatmap_data(
        results_high, mode="all", subject_id=patient, target_class=1)

    plot_global_heatmap_and_bars(
        heatmap_right, results_high,
        title_suffix=f"Patient {patient} - Right Hand Only",
        channel_names=ch_names,
        save_path=right_hand_path,
        show_plot=False,
    )

    # 3. Difference
    plot_difference_heatmap_and_bars(
        heatmap_right - heatmap_left,
        analysis_results=results_high,
        title_suffix=f"Patient {patient} - Right Left Difference",
        channel_names=ch_names,
        save_path=diff_path,
        show_plot=False,
    )

In [ ]:
# 13. Wykresy globalne: całkowity wpływ / wsparcie poprawnej klasy / konflikt
# Total absolute network attention
heatmap_all = extract_global_heatmap_data(results_high, mode="all")
plot_global_heatmap_and_bars(
    heatmap_all, results_high,
    title_suffix="Global / Total Impact",
    channel_names=ch_names,
    save_path=os.path.join(base_output_dir, "Global.png"),
    show_plot=False,
)

# Attention pointing TOWARDS the correct classification
heatmap_correct = extract_global_heatmap_data(results_high, mode="correct_direction")
plot_global_heatmap_and_bars(
    heatmap_correct, results_high,
    title_suffix="Correct Class Support",
    channel_names=ch_names,
    save_path=os.path.join(base_output_dir, "Global Correct.png"),
    show_plot=False,
)

# Attention pointing AWAY from the correct classification (Conflict/Noise)
heatmap_wrong = extract_global_heatmap_data(results_high, mode="incorrect_direction")
plot_global_heatmap_and_bars(
    heatmap_wrong, results_high,
    title_suffix="Incorrect Class Influence (Noise/Error)",
    channel_names=ch_names,
    save_path=os.path.join(base_output_dir, "Global Incorrect.png"),
    show_plot=False,
)

In [ ]:
# 14. Sprzątanie Colaba (zawsze na końcu: zwalnia runtime)
from google.colab import drive, runtime

drive.flush_and_unmount()
runtime.unassign()